In [4]:
import pandas as pd

data = {
    "user_id": [1, 1, 2, 2],
    "timestamp": [
        "2026-09-18 10:30:00",
        "2026-09-18 02:15:00",
        "2026-09-19 14:20:00",
        "2026-09-20 23:45:00"
    ],
    "status": [
        "SUCCESS",
        "FAILED",
        "SUCCESS",
        "FAILED"
    ]
}

df = pd.DataFrame(data)

print(df)

   user_id            timestamp   status
0        1  2026-09-18 10:30:00  SUCCESS
1        1  2026-09-18 02:15:00   FAILED
2        2  2026-09-19 14:20:00  SUCCESS
3        2  2026-09-20 23:45:00   FAILED


In [5]:
# Feature engineering

df["hour"] = pd.to_datetime(df["timestamp"]).dt.hour
df["day_of_week"] = pd.to_datetime(df["timestamp"]).dt.dayofweek
df["is_weekend"] = df['day_of_week'].isin([5, 6]).astype(int)

print(df)

   user_id            timestamp   status  hour  day_of_week  is_weekend
0        1  2026-09-18 10:30:00  SUCCESS    10            4           0
1        1  2026-09-18 02:15:00   FAILED     2            4           0
2        2  2026-09-19 14:20:00  SUCCESS    14            5           1
3        2  2026-09-20 23:45:00   FAILED    23            6           1


In [6]:
import pandas as pd

data = {
    "user_id": [1, 1, 2, 2, 2, 3],
    "timestamp": [
        "2026-09-18 10:30:00",
        "2026-09-18 10:35:00",
        "2026-09-18 11:00:00",
        "2026-09-18 11:05:00",
        "2026-09-18 11:10:00",
        "2026-09-18 12:00:00"
    ],
    "status": [
        "SUCCESS",
        "FAILED",
        "SUCCESS",
        "SUCCESS",
        "FAILED",
        "SUCCESS"
    ]
}

df = pd.DataFrame(data)

print(df)

   user_id            timestamp   status
0        1  2026-09-18 10:30:00  SUCCESS
1        1  2026-09-18 10:35:00   FAILED
2        2  2026-09-18 11:00:00  SUCCESS
3        2  2026-09-18 11:05:00  SUCCESS
4        2  2026-09-18 11:10:00   FAILED
5        3  2026-09-18 12:00:00  SUCCESS


In [7]:
# using groupby() in feature engineering

grouped = df.groupby("user_id")

print(grouped)

In [8]:
# counting the queries

query_count = df.groupby("user_id").size()

print(query_count)

user_id
1    2
2    3
3    1
dtype: int64


In [9]:
query_count = df.groupby("user_id").size().reset_index(name="total_queries")

print(query_count)

   user_id  total_queries
0        1              2
1        2              3
2        3              1


In [11]:
# Creating a feature for failed queries

df["failed"] = (df["status"] == "FAILED").astype(int)

print(df)

   user_id            timestamp   status  failed
0        1  2026-09-18 10:30:00  SUCCESS       0
1        1  2026-09-18 10:35:00   FAILED       1
2        2  2026-09-18 11:00:00  SUCCESS       0
3        2  2026-09-18 11:05:00  SUCCESS       0
4        2  2026-09-18 11:10:00   FAILED       1
5        3  2026-09-18 12:00:00  SUCCESS       0


In [12]:
failed_count = df.groupby("user_id")["failed"].sum()

print(failed_count)

user_id
1    1
2    1
3    0
Name: failed, dtype: int64


In [13]:
# Pandas lets us do multiple calculations together using: groupby().agg()

user_stats = df.groupby("user_id").agg(
    total_queries=("user_id", "count"),
    failed_queries=("failed", "sum")
)

print(user_stats)

         total_queries  failed_queries
user_id                               
1                    2               1
2                    3               1
3                    1               0


In [16]:
import pandas as pd

data = {
    "user_id": [1, 1, 1, 2, 2, 3],
    "execution_time_ms": [100, 150, 200, 300, 500, 120],
    "status": [
        "SUCCESS",
        "SUCCESS",
        "FAILED",
        "SUCCESS",
        "FAILED",
        "SUCCESS"
    ]
}

df = pd.DataFrame(data)

print(df)

   user_id  execution_time_ms   status
0        1                100  SUCCESS
1        1                150  SUCCESS
2        1                200   FAILED
3        2                300  SUCCESS
4        2                500   FAILED
5        3                120  SUCCESS


In [17]:
# aggregation : calculate some summary information for each group.

result = df.groupby("user_id").agg({
    "execution_time_ms": "mean"
})

print(result)

         execution_time_ms
user_id                   
1                    150.0
2                    400.0
3                    120.0


In [18]:
# asking for maximum exection time.

result = df.groupby("user_id").agg({
    "execution_time_ms": ["mean", "max"]
})

print(result)

        execution_time_ms     
                     mean  max
user_id                       
1                   150.0  200
2                   400.0  500
3                   120.0  120


In [33]:
result = df.groupby("user_id").agg({
    "execution_time_ms": ["mean", "max", "count"]
})

print(result)

        execution_time_ms           
                     mean  max count
user_id                             
1                   150.0  200     3
2                   400.0  500     2
3                   120.0  120     1


In [35]:
# adding failed queries

df["failed"] = (df["status"] == "FAILED").astype(int)

result = df.groupby("user_id").agg({
    "execution_time_ms": ["mean", "max", "count"],
    "failed": "sum"
})

print(result)

        execution_time_ms            failed
                     mean  max count    sum
user_id                                    
1                   150.0  200     3      1
2                   400.0  500     2      1
3                   120.0  120     1      0


In [36]:
# Cleaning up the column names

user_stats = df.groupby("user_id").agg({
    "execution_time_ms": ["mean", "max", "count"],
    "failed": "sum"
}).reset_index()

user_stats.columns = [
    "user_id",
    "avg_execution_time",
    "max_execution_time",
    "total_queries",
    "failed_queries"
]

print(user_stats)

   user_id  avg_execution_time  max_execution_time  total_queries  \
0        1               150.0                 200              3   
1        2               400.0                 500              2   
2        3               120.0                 120              1   

   failed_queries  
0               1  
1               1  
2               0  


In [37]:
# Connecting all things to ML

X = user_stats[
    [
        "avg_execution_time",
        "max_execution_time",
        "total_queries",
        "failed_queries"
    ]
].values # .values gives us the numerical matrix that scikit-learn expects.

In [38]:
# Isolation Forest

from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(X)

predictions = model.predict(X)
scores = model.score_samples(X)

user_stats["prediction"] = predictions
user_stats["anomaly_score"] = scores

print(user_stats)

   user_id  avg_execution_time  max_execution_time  total_queries  \
0        1               150.0                 200              3   
1        2               400.0                 500              2   
2        3               120.0                 120              1   

   failed_queries  prediction  anomaly_score  
0               1           1      -0.339839  
1               1           1      -0.390042  
2               0          -1      -0.427566  


In [39]:
# Converting prediction into an alert
user_stats["is_anomaly"] = user_stats["prediction"] == -1
anomalies = user_stats[user_stats["is_anomaly"]]

print(anomalies)

   user_id  avg_execution_time  max_execution_time  total_queries  \
2        3               120.0                 120              1   

   failed_queries  prediction  anomaly_score  is_anomaly  
2               0          -1      -0.427566        True  
